# v5_deepsets_pinn — entrenamiento en Google Colab (GitHub-based)

Igual flujo que `v4_deepsets_colab.ipynb` pero entrena **v5** — DeepSets con loss físicas (divergencia + rotor de Maxwell) sobre el campo B completo (Bx, By, Bz).

Loss = `MSE_data + λ_div · (∇·B)² + λ_curl · |∇×B|²`. `λ_div` y `λ_curl` arrancan en `1e-3`, se pueden subir si las losses físicas se quedan altas.

## Setup previo (una sola vez)
1. Subí el HDF5 a Google Drive en `MyDrive/Ipre/datasets/v2_xyz100_step50_n5000.h5` (mismo dataset que v4).
2. Colab → ícono 🔑 *Secrets* → agregá `COMET_API_KEY`. Activá *Notebook access*.
3. *Runtime → Change runtime type → GPU*.

## Cada sesión
Corré las celdas de arriba a abajo.

## 1. GPU

In [ ]:
!nvidia-smi

## 2. Clonar el repo desde GitHub

In [ ]:
REPO_URL = 'https://github.com/Daspony/PMDKernel.git'
BRANCH   = 'main'

import os
if os.path.isdir('PMDKernel'):
    %cd PMDKernel
    !git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL}
    %cd PMDKernel
print('cwd:', os.getcwd())
!git log -1 --oneline

## 3. Instalar dependencias

Colab ya trae `torch` con CUDA. Solo falta el resto.

In [ ]:
!pip install -q pytorch-lightning comet-ml h5py torchmetrics

In [ ]:
import torch, pytorch_lightning as pl, comet_ml, h5py, sklearn, torchmetrics
print(f'torch        = {torch.__version__}  cuda={torch.cuda.is_available()}')
print(f'lightning    = {pl.__version__}')
print(f'comet_ml     = {comet_ml.__version__}')
print(f'h5py         = {h5py.__version__}')
print(f'torchmetrics = {torchmetrics.__version__}')
if torch.cuda.is_available():
    print(f'device       = {torch.cuda.get_device_name(0)}')

## 4. Bajar el dataset desde Drive

Monto Drive solo para copiar el H5 al directorio local del runtime. DespuÃ©s podÃ©s desmontar si querÃ©s.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATASET = 'v2_xyz100_step50_n5000.h5'
DRIVE_SRC = f'/content/drive/MyDrive/Ipre/datasets/{DATASET}'

!mkdir -p data/datasets
!cp -v {DRIVE_SRC} data/datasets/
!ls -lh data/datasets/

## 5. Credenciales Comet

Lee `COMET_API_KEY` del panel de Secrets de Colab. Si lo dejÃ¡s vacÃ­o, podÃ©s desactivar el logger pasando `os.environ['COMET_OFFLINE'] = '1'`.

In [ ]:
from google.colab import userdata
import os

os.environ['COMET_API_KEY']   = userdata.get('COMET_API_KEY')
os.environ['COMET_WORKSPACE'] = 'sebasti-n-vallejos'
print('comet workspace:', os.environ['COMET_WORKSPACE'])
print('api key len:', len(os.environ['COMET_API_KEY']))

## 6. Entrenar

Editá `RUN_TAG`, `EPOCHS`, `LAMBDA_DIV`, `LAMBDA_CURL` si querés.

In [ ]:
H5            = f'data/datasets/{DATASET}'
RUN_TAG       = 'v5_pinn_v2_n5000_colab'
EPOCHS        = 100
PATIENCE      = 20
BATCH_SIZE    = 16     # configs por step. T4=16-32, A100/cluster=64-128
LAMBDA_DIV    = 1e-3
LAMBDA_CURL   = 1e-3
COMET_PROJECT = 'pmdkernel'
CKPT_DIR      = f'/content/drive/MyDrive/Ipre/runs/{RUN_TAG}'

!python python/train_v5.py \
    --h5 {H5} \
    --run-tag {RUN_TAG} \
    --ckpt-dir {CKPT_DIR} \
    --epochs {EPOCHS} \
    --patience {PATIENCE} \
    --batch-size {BATCH_SIZE} \
    --lambda-div {LAMBDA_DIV} \
    --lambda-curl {LAMBDA_CURL} \
    --comet-project {COMET_PROJECT} \
    --no-progress \
    --num-workers 2 \
    --pin-memory

### Smoke test (3 epochs)

In [ ]:
!python python/train_v5.py \
    --h5 data/datasets/{DATASET} \
    --run-tag v5_pinn_colab_smoke \
    --quick \
    --lambda-div 1e-3 \
    --lambda-curl 1e-3 \
    --no-progress \
    --num-workers 2 \
    --pin-memory

## 7. Verificar que el ckpt persistió en Drive

Como pasamos `--ckpt-dir` apuntando directo a Drive, Lightning escribió `<run_tag>.ckpt`, `last.ckpt` y `<run_tag>_aux.pt` ahí durante el train. No hace falta `cp` al final.

In [ ]:
!ls -lh {CKPT_DIR}/

## 8. Inspeccionar resultados

In [ ]:
import torch
from pathlib import Path

aux_path = Path(CKPT_DIR) / f'{RUN_TAG}_aux.pt'
aux = torch.load(aux_path, weights_only=False)

print('checkpoint:', aux['ckpt_path'])
print('comet url :', aux['comet_url'])
print()
for split, m in aux['metrics'].items():
    print(f"{split:5s}  rmse={m['rmse_mt']:.4f} mT  r2={m['r2']:.4f}")

## 9. Inferencia sobre una muestra del test split

v5 predice **el campo B completo (Bx, By, Bz)**, no solo By. Los reportes incluyen RMSE por componente.

In [ ]:
import sys, os, torch, numpy as np, h5py
from pathlib import Path

PROJECT = '/content/PMDKernel'
os.chdir(PROJECT)

for cand in ('python', 'Python'):
    p = os.path.join(PROJECT, cand)
    if os.path.isdir(p):
        if p not in sys.path:
            sys.path.insert(0, p)
        break
else:
    raise RuntimeError('no se encontro python/ ni Python/')

from Models.v5_deepsets_pinn.model import LitDeepSetPINN

ckpt_path = Path(CKPT_DIR) / f'{RUN_TAG}.ckpt'
if not ckpt_path.exists():
    cands = sorted(Path(CKPT_DIR).glob(f'{RUN_TAG}*.ckpt'))
    cands = [c for c in cands if 'last' not in c.name]
    ckpt_path = cands[-1]
aux_path = Path(CKPT_DIR) / f'{RUN_TAG}_aux.pt'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = LitDeepSetPINN.load_from_checkpoint(str(ckpt_path), map_location=device, weights_only=False).eval().to(device)
aux    = torch.load(aux_path, map_location='cpu', weights_only=False)

print(f"ckpt         : {ckpt_path.name}")
print(f"device       : {device}")
print(f"train h5     : {aux['h5_path']}")
print(f"sensor_xyz   : {aux['sensor_xyz'].shape}")
print(f"b_mean (Bx,By,Bz) : {np.asarray(aux['b_mean'])}  mT")
print(f"b_std  (Bx,By,Bz) : {np.asarray(aux['b_std'])}  mT")
m_te = aux['metrics']['test']
print(f"training test     : RMSE={m_te['rmse_mt']:.4e} mT  R2={m_te['r2']:+.4f}")

In [ ]:
# Reconstruir el split test del training para tomar un sample que el modelo NO vio
from Models.v5_deepsets_pinn import data as data_mod

H5_FULL = 'data/datasets/' + DATASET
ds      = data_mod.load_dataset_metadata(H5_FULL)
splits  = data_mod.split_indices(ds['N'], val_frac=0.15, test_frac=0.15, seed=42)
test_idx = splits['test']
print(f"n_test = {len(test_idx)} muestras  (sample ids: {test_idx[:5]}...)")

TEST_POS = 0  # 0..n_test-1 - cambia para mirar otra muestra
sample_id = int(test_idx[TEST_POS])

with h5py.File(H5_FULL, 'r') as f:
    sensors_by = f['geometria/sens/B'][sample_id][..., 1].astype(np.float32)  # (I,) mT, solo By en sensores
    R_grid     = f['geometria/grid/R'][:].astype(np.float32)                    # (J, 3) mm
    B_true     = f['geometria/grid/B'][sample_id].astype(np.float32)            # (J, 3) mT — full B
    gx = f['geometria/grid/meta/x'][:].astype(np.float32)
    gy = f['geometria/grid/meta/y'][:].astype(np.float32)
    gz = f['geometria/grid/meta/z'][:].astype(np.float32)

Nx, Ny, Nz = len(gx), len(gy), len(gz)
J = R_grid.shape[0]
I = sensors_by.shape[0]
print(f"sample id     : {sample_id}  (pos {TEST_POS} en test split)")
print(f"grid          : {Nx}x{Ny}x{Nz} = {J} puntos")
print(f"sensores By   : min={sensors_by.min():.3f}  max={sensors_by.max():.3f}  std={sensors_by.std():.3f}  mT")
for k, name in enumerate(('Bx', 'By', 'Bz')):
    print(f"{name} true       : min={B_true[:,k].min():+.3f}  max={B_true[:,k].max():+.3f}  mean={B_true[:,k].mean():+.3f}  mT")

In [ ]:
# Forward sobre los J puntos. v5 devuelve (J, 3) con (Bx, By, Bz) en mT.
sensors_rep = np.broadcast_to(sensors_by, (J, I)).copy()   # (J, I)
sensors_t   = torch.from_numpy(sensors_rep).to(device)
query_t     = torch.from_numpy(R_grid).to(device)

with torch.no_grad():
    B_pred = model.predict_mT(sensors_t, query_t).cpu().numpy()    # (J, 3)

err  = B_pred - B_true                                              # (J, 3)
err_mag = np.linalg.norm(err, axis=1)                               # (J,)
true_mag = np.linalg.norm(B_true, axis=1)

rmse_total = float(np.sqrt(np.mean(err**2)))
mae_total  = float(np.abs(err).mean())
maxe_total = float(err_mag.max())
r2_total   = float(1 - (err**2).sum() / ((B_true - B_true.mean(axis=0))**2).sum())

print(f"GLOBAL (3 comp):")
print(f"  RMSE      : {rmse_total:.4e} mT")
print(f"  MAE       : {mae_total:.4e} mT")
print(f"  max |err| : {maxe_total:.4e} mT   ({100*maxe_total/true_mag.max():.2f}% del |B| max)")
print(f"  R2        : {r2_total:+.4f}")
print()
print('POR COMPONENTE:')
for k, name in enumerate(('Bx', 'By', 'Bz')):
    e_k = err[:, k]
    rmse_k = float(np.sqrt(np.mean(e_k**2)))
    r2_k   = float(1 - (e_k**2).sum() / ((B_true[:,k] - B_true[:,k].mean())**2).sum())
    print(f'  {name}: RMSE={rmse_k:.4e} mT   R2={r2_k:+.4f}')
print()
print('idx   Bx_true   By_true   Bz_true   Bx_pred   By_pred   Bz_pred')
for j in range(min(J, 10)):
    print(f"{j:3d}   " + "   ".join(f"{B_true[j,k]:+.3f}" for k in range(3)) + "   " +
          "   ".join(f"{B_pred[j,k]:+.3f}" for k in range(3)))

## 10. Verificación física de la predicción (autograd)

Computa `|∇·B|` y `|∇×B|` del **campo predicho** vía autograd (mismo método que se usó en training). Si el modelo aprendió a respetar Maxwell, ambos deberían estar cerca de cero.

Unidades: T/m (= mT/mm, ver `model.py:20-21`).

_Reutiliza `model`, `sensors_by`, `R_grid`, `I`, `J` de las celdas 22 y 23._

In [ ]:
# Reusamos sensors_by + R_grid de la celda 23 (mismo sample)
sensors_t = torch.from_numpy(np.broadcast_to(sensors_by, (J, I)).copy()).to(device)
query_t   = torch.from_numpy(R_grid).to(device)

# Autograd requiere requires_grad sobre las coords
with torch.enable_grad():
    q_g = query_t.detach().clone().requires_grad_(True)
    b_norm, div, cx, cy, cz = model._physics_residuals(sensors_t, q_g)
    curl_mag = torch.sqrt(cx**2 + cy**2 + cz**2)

div_np  = div.detach().cpu().numpy()        # (J,) en T/m
curl_np = curl_mag.detach().cpu().numpy()   # (J,) en T/m

print(f"Sample id: {sample_id}  ({J} puntos)")
print()
print("|div B|  [T/m]")
print(f"  mean = {np.abs(div_np).mean():.3e}")
print(f"  p50  = {np.percentile(np.abs(div_np), 50):.3e}")
print(f"  p95  = {np.percentile(np.abs(div_np), 95):.3e}")
print(f"  max  = {np.abs(div_np).max():.3e}")
print()
print("|curl B|  [T/m]")
print(f"  mean = {curl_np.mean():.3e}")
print(f"  p50  = {np.percentile(curl_np, 50):.3e}")
print(f"  p95  = {np.percentile(curl_np, 95):.3e}")
print(f"  max  = {curl_np.max():.3e}")
print()
print("Detalle por punto (primeros 10):")
print("idx     div         |curl|       (T/m)")
for j in range(min(J, 10)):
    print(f"{j:3d}   {div_np[j]:+.3e}   {curl_np[j]:.3e}")

# Worst offenders
print()
print("Top 5 puntos con |div| mas alto:")
worst = np.argsort(-np.abs(div_np))[:5]
for j in worst:
    x, y, z = R_grid[j]
    print(f"  j={j:3d}  xyz=({x:+.0f},{y:+.0f},{z:+.0f}) mm   div={div_np[j]:+.3e}   |curl|={curl_np[j]:.3e}  T/m")